# MolGap conservative 2D + 3D Fusion
Drive-backed six-head screen with exact 2D fallback and fixed OOD/P8-hard acceptance.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import subprocess
import sys

ROOT = Path('/content/drive/MyDrive/MolGap')
NOTEBOOK_DIR = ROOT / 'notebooks' / 'molgap_conservative_2d3d_fusion_r1'
INPUT_DIR = ROOT / 'results' / 'molgap_conservative_2d3d_fusion_r1' / 'inputs'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'molgap_conservative_2d3d_fusion_r1'
RESULTS_DIR = ROOT / 'results' / 'molgap_conservative_2d3d_fusion_r1' / 'run'
WHEEL = NOTEBOOK_DIR / 'molgap-0.1.0-py3-none-any.whl'
EXPECTED_WHEEL_SHA256 = '8f17c61e077e44556758d5b65898440236b6fd16dbf96be13fd46e189ae0c6c8'

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not WHEEL.is_file():
    raise FileNotFoundError(WHEEL)
if sha256_file(WHEEL) != EXPECTED_WHEEL_SHA256:
    raise RuntimeError('The uploaded wheel differs from the notebook contract')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', str(WHEEL)],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 GPU runtime before continuing')
GPU_NAME = torch.cuda.get_device_name(0)
if 'A100' not in GPU_NAME:
    raise RuntimeError(f'A100 required by this run contract; active GPU is {GPU_NAME}')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', GPU_NAME)
print('checkpoints:', CHECKPOINT_DIR)
print('results:', RESULTS_DIR)


In [ ]:
import json

TRAINING_PAYLOAD = INPUT_DIR / 'training_payload.pt'
TRAINING_MANIFEST = INPUT_DIR / 'training_manifest.json'
EXTERNAL_PAYLOAD = INPUT_DIR / 'external_payload.pt'
EXTERNAL_MANIFEST = INPUT_DIR / 'external_manifest.json'

def accept_payload(payload_path, manifest_path):
    if not payload_path.is_file():
        raise FileNotFoundError(payload_path)
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    manifest = json.loads(manifest_path.read_text())
    expected = manifest.get('payload', {}).get('sha256')
    observed = sha256_file(payload_path)
    if manifest.get('status') != 'accepted' or observed != expected:
        raise RuntimeError(f'Payload acceptance failed: {payload_path.name}')
    return manifest

training_manifest = accept_payload(TRAINING_PAYLOAD, TRAINING_MANIFEST)
external_manifest = accept_payload(EXTERNAL_PAYLOAD, EXTERNAL_MANIFEST)
if training_manifest['context_dim'] != external_manifest['context_dim']:
    raise RuntimeError('Training and external context dimensions differ')
if external_manifest['scope_rows'] != {
    'all': 1973,
    'ood1000': 998,
    'p8_targeted_hard': 975,
}:
    raise RuntimeError('Fixed external evaluation identity differs')
print('training rows:', training_manifest['rows'])
print('external rows:', external_manifest['scope_rows'])
print('context dim:', training_manifest['context_dim'])
print('sealed 20K: not mounted and not used')


In [ ]:
from molgap.conservative_fusion_runner import run_conservative_fusion

result = run_conservative_fusion(
    training_payload_path=TRAINING_PAYLOAD,
    external_payload_path=EXTERNAL_PAYLOAD,
    checkpoint_dir=CHECKPOINT_DIR,
    results_dir=RESULTS_DIR,
    device='cuda',
)
print('decision:', result['status'])
print('elapsed hours:', result['elapsed_s'] / 3600)
print('promotion gate:', json.dumps(result['promotion_gate'], indent=2))
print('completion:', RESULTS_DIR / 'completion_manifest.json')


In [ ]:
summary = json.loads((RESULTS_DIR / 'metrics.json').read_text())
print('decision:', summary['status'])
for base, gate in summary['promotion_gate'].items():
    print(
        base,
        'all delta=', f"{gate['all_average_delta_eV']:+.6f} eV",
        'P8-hard delta=', f"{gate['p8_hard_average_delta_eV']:+.6f} eV",
        'PASS' if gate['passed'] else 'REJECT',
    )
print('No production registry was changed.')
